# Change a model

The model is `examples/dispatch.yaml` — least-cost generation against a load
profile. Nothing here mutates it: every cell is a function of a **spec** and its
**sources**, so cells re-run in any order mean the same thing, and what you
carry out of the session is a file rather than a kernel.

Three loops, cheapest first:

1. **new numbers** — `rebind`, and the solver keeps the model it has loaded
2. **more rows** — the same math over a longer axis, which is still data
3. **new math** — the spec is a `dict`; patch it and re-run

What none of them is: `model.add_constraint(...)`. There is no Python API for
building a model here, on purpose — see the last cell.

In [ ]:
import polars as pl
from IPython.display import Markdown

import lpspec as lps

MODEL = '../examples/dispatch.yaml'
GENERATORS = ['wind', 'solar', 'gas']

sources = {
    'p_max': pl.DataFrame({'generator': GENERATORS, 'value': [80.0, 40.0, 200.0]}),
    'cost': pl.DataFrame({'generator': GENERATORS, 'value': [0.0, 0.0, 60.0]}),
    'load': pl.DataFrame({'snapshot': range(6), 'value': [90.0, 120.0, 150.0, 180.0, 140.0, 100.0]}),
}

Markdown(lps.to_markdown(MODEL))

## 1. New numbers

`build` once, `rebind` per run of the cell. The declarations are untouched, so
the model HiGHS holds is untouched too: new costs go onto it in place and the
next solve starts from the basis the last one ended on.

In [ ]:
bound = lps.build(MODEL, sources)

rows = []
for gas_cost in (40.0, 60.0, 90.0):
    costs = pl.DataFrame({'generator': GENERATORS, 'value': [0.0, 0.0, gas_cost]})
    rows.append({'gas_cost': gas_cost, 'objective': bound.rebind({'cost': costs}).solve().objective})

sweep = pl.DataFrame(rows)
warm = bound.diagnostics()

print(f'{warm.loads} model loaded, {warm.solves} solves')
sweep

`loads` is 1 against `solves` of 3: three answers, one model ever loaded.
That counter is the difference between "lpspec is slow" and "this loop is
rebuilding every time", and nothing about the answers depends on it.

The answers themselves are held to an equality: `bound.rebind(x).solve()` gives
what `lps.solve(MODEL, sources | x)` gives, always. It is the oracle to reach
for when a loop looks wrong.

In [ ]:
costs = pl.DataFrame({'generator': GENERATORS, 'value': [0.0, 0.0, 90.0]})
fresh = lps.solve(MODEL, sources | {'cost': costs}).objective
rebound = sweep.filter(pl.col('gas_cost') == 90.0).item(0, 'objective')

print(f'rebound {rebound:,.1f} — fresh build {fresh:,.1f}')

## 2. More rows

A longer horizon is not a different model — it is a longer table plus the
`coords=` to match. The same move grows a Benders cut family one cut at a time
(`examples/benders/run.py`), which is what makes "add a constraint" a data
question far more often than it looks.

In [ ]:
horizon = pl.DataFrame(
    {
        'snapshot': range(12),
        'value': [90.0, 120.0, 150.0, 180.0, 140.0, 100.0, 95.0, 130.0, 160.0, 190.0, 150.0, 110.0],
    }
)

answer = bound.rebind({'load': horizon}, coords={'snapshot': range(12)}).solve()
grown = bound.diagnostics()
schedule = answer.primal('p')

print(f'{schedule.height} rows of p now, and {grown.loads} loads over {grown.solves} solves')
schedule.head()

`loads` is 2 now. New coordinates renumber the columns, so that model was
loaded from scratch and solved cold — the fast path is what changes, never the
answer. Read the counter, not the clock.

`p` comes back tidy and **in label order** — one row per built variable,
`(snapshot, generator, value)`, row-major over the variable's coordinate
product, so two reads and two runs agree and nothing here has to sort.
`schedule.pivot(on='generator', index='snapshot', values='value')` is the wide
view if you would rather read a schedule than a table of rows.

## 3. New math

`to_dict()` is the model as data, and every verb takes a `dict` as readily as a
path. So an edit is a key, and the round trip through `load_model` re-validates
it — the language's load-time errors are this notebook's error messages.

Below: a ramp limit on gas, which needs a parameter as well as a constraint.

In [ ]:
spec = lps.load_model(MODEL).to_dict()
spec['parameters']['ramp_max'] = {'dims': ['generator']}
spec['constraints']['ramp_up'] = {
    'foreach': ['snapshot', 'generator'],
    'expression': 'p - shift(p, over=snapshot, by=1) <= ramp_max',
}

ramp_max = pl.DataFrame({'generator': GENERATORS, 'value': [100.0, 100.0, 20.0]})
base = lps.solve(MODEL, sources).objective
ramped = lps.solve(spec, sources | {'ramp_max': ramp_max}).objective

pl.DataFrame({'model': ['dispatch', 'dispatch + ramp limit'], 'objective': [base, ramped]})

The limit binds: gas cannot reach the evening peak in one step, so it starts
climbing early and free wind is curtailed to make room for it.

The math re-renders from the patched spec, which is the check that the edit
says what you meant:

In [ ]:
Markdown(lps.to_markdown(spec, legend=False, numbered=False))

An edit the language refuses is refused before any data is bound — `check`
parses, resolves and lowers, and nothing else runs:

In [ ]:
typo = {
    **spec,
    'constraints': {
        **spec['constraints'],
        'peak': {'foreach': ['snapshot'], 'expression': 'sum(p, over=generators) <= load'},
    },
}

try:
    lps.check(typo)
except lps.LanguageError as exc:
    print(exc)

## What leaves the session

The spec, as the file you diff against `examples/dispatch.yaml` and commit —
not this notebook, and not a pickle of the kernel. A model built as a `dict` still gets
a file, which is the whole point of the round trip:

In [ ]:
print(lps.load_model(spec).to_yaml())

## What this notebook cannot do

Mutating a *built* model — `fix`, `relax`, deleting a constraint — printing a
single row, and an IIS on an infeasible one. Those are linopy's lifecycle and
debugging verbs, they are genuinely ahead of anything here, and the first of
them is refused by design rather than unimplemented: the public interface is a
declared model, not a Python API
([hard rule 5](https://github.com/fluxopt/lpspec/blob/main/docs/ARCHITECTURE.md#hard-rules)).

What that buys is above — a cell that cannot half-apply, a session whose output
is a diff, and re-run order that cannot change what the model means. The whole
relationship, including where linopy is ahead, is
[docs/design/linopy.md](https://github.com/fluxopt/lpspec/blob/main/docs/design/linopy.md)
and the
[honest snapshot](https://github.com/fluxopt/lpspec/blob/main/docs/ROADMAP.md#honest-snapshot).